# Temporal aggregation: from high to low frequency

In [1]:
# Setup: install the toolkit when running on Google Colab, and a plotting style
import sys

if "google.colab" in sys.modules:
    %pip install -q "econts[book] @ git+https://github.com/hmorao95/econ-time-series-manual"

import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

Going from daily to monthly, or monthly to quarterly, is a one-liner with
`resample`. The economics is in choosing the **aggregation rule**, which
depends on what the series measures.

In [2]:
import numpy as np
import pandas as pd
from econts import load

retail = load("RSXFS")  # monthly flow, $ millions
payems = load("PAYEMS")  # monthly employment level
unrate = load("UNRATE")  # monthly rate
brent = load("DCOILBRENTEU")  # daily price
claims = load("ICSA")  # weekly flow, weeks ending Saturday

## The basic pattern

`resample` takes a frequency alias (chapter 1) and an aggregation. With
start-of-period aliases (`QS`, `YS`) the result keeps FRED's labelling; with
`PeriodIndex` input you get periods back.

In [3]:
q = pd.DataFrame(
    {
        "retail (sum)": retail.resample("QS").sum(min_count=3),
        "payrolls (mean)": payems.resample("QS").mean(),
        "payrolls (end)": payems.resample("QS").last(),
        "unrate (mean)": unrate.resample("QS").mean(),
    }
)
print(q.loc["2025":].round(1))

# Same thing on periods
print(unrate.to_period("M").resample("Q").mean().tail(2))

            retail (sum)  payrolls (mean)  payrolls (end)  unrate (mean)
date                                                                    
2025-01-01     1812501.0         158318.3        158377.0            4.1
2025-04-01     1814141.0         158487.0        158478.0            4.2
2025-07-01     1839756.0         158520.7        158548.0            4.3
2025-10-01     1851051.0         158429.7        158432.0            4.4
2026-01-01     1867151.0         158559.3        158650.0            4.3
2026-04-01     1919897.0         158850.3        158892.0            4.3
2026-07-01     1938818.0         158980.3        159044.0            4.1
date
2026Q2    4.266667
2026Q3    4.133333
Freq: Q-DEC, Name: UNRATE, dtype: float64


`sum` returns 0 for an empty period and would happily sum two months of a
three-month quarter. `min_count=3` returns `NaN` instead, which is what you
want for the last, incomplete quarter.

## Incomplete periods

A general guard: aggregate the observation count alongside the value, and mask
periods with too few observations.

In [4]:
def aggregate(s: pd.Series, freq: str, how: str = "mean", min_obs: int = 1):
    """Resample `s` with `how`, setting periods with < `min_obs` values to NaN."""
    r = s.resample(freq)
    out = r.agg(how)
    return out.where(r.count() >= min_obs)


brent_m = aggregate(brent, "MS", "mean", min_obs=15)
print(brent_m.tail(3).round(2))  # current month is NaN until 15 trading days exist

date
2026-09-01    114.16
2026-10-01    123.93
2026-11-01       NaN
Freq: MS, Name: DCOILBRENTEU, dtype: float64


```{admonition} In your projects
:class: tip
The same recipe is in the `econts` toolkit:

    from econts.aggregate import aggregate
```

For daily prices, also choose the **concept**: the monthly average
(`mean`), the end-of-month price (`last`), or the full open/high/low/close:

In [5]:
print(brent.resample("MS").ohlc().loc["2026"].round(2).tail(3))

              open    high     low   close
date                                      
2026-09-01   96.02  130.80   96.02  115.91
2026-10-01  114.82  135.51  114.82  120.15
2026-11-01  107.39  107.39  107.03  107.03


## Stocks: end of period vs average

Payroll employment is a level; the quarterly figure can be the average of the
three months (BLS and BEA convention) or the March/June/September/December
value. The difference matters around turning points:

In [6]:
gap = q["payrolls (end)"] - q["payrolls (mean)"]
print(gap["2020"].round(0))  # thousands: about 4 million jobs in 2020Q2

date
2020-01-01    -845.0
2020-04-01    3959.0
2020-07-01    1155.0
2020-10-01     -32.0
Freq: QS-JAN, dtype: float64


## SAAR flows

US national-accounts flows are annualized. The annual value is the **mean**
of the four quarters, not the sum (the sum would be four times too large).

In [7]:
gdp = load("GDPC1")
print(gdp.resample("YS").mean().loc["2023":"2025"].round(0))

date
2023-01-01    22724.0
2024-01-01    23395.0
2025-01-01    23943.0
Freq: YS-JAN, Name: GDPC1, dtype: float64


For non-annualized quarterly flows (most European national accounts), sum.

## Weekly to monthly

Weeks straddle month boundaries. Resampling by the week-ending date puts
all of the week Sunday 30 March to Saturday 5 April into April. For a flow
like jobless claims, spread each week over its seven days and sum by month:

In [8]:
def weekly_flow_to_monthly(w: pd.Series) -> pd.Series:
    """Distribute each weekly total evenly over its days, then sum by month."""
    days = pd.date_range(w.index[0] - pd.Timedelta(days=6), w.index[-1], freq="D")
    daily = (w / 7).reindex(days, method="bfill")  # each day gets 1/7 of its week
    monthly = daily.resample("MS").sum()
    return monthly.iloc[1:-1]  # drop partial first/last months


naive = claims.resample("MS").sum()
exact = weekly_flow_to_monthly(claims)
print(
    pd.DataFrame({"by week-end date": naive, "day-weighted": exact})
    .loc["2025-01":"2025-06"]
    .round(-3)
)

            by week-end date  day-weighted
2025-01-01          860000.0      960000.0
2025-02-01          900000.0      903000.0
2025-03-01         1115000.0      987000.0
2025-04-01          903000.0      970000.0
2025-05-01         1159000.0     1029000.0
2025-06-01          956000.0     1021000.0


```{admonition} In your projects
:class: tip
The same recipe is in the `econts` toolkit:

    from econts.aggregate import weekly_to_monthly
```

For a weekly *average* concept (a rate, a price), use the day-weighted
`mean` instead of `sum`.

## Fiscal years and custom periods

Anchored aliases do fiscal years directly. A fiscal year ending in September
(the US federal year) labelled by the year in which it ends:

In [9]:
fy = retail.resample("YE-SEP").sum(min_count=12)
print(fy.tail(3).round(-3))

date
2024-09-30    7036000.0
2025-09-30    7275000.0
2026-09-30    7577000.0
Freq: YE-SEP, Name: RSXFS, dtype: float64


Any custom grouping works with `groupby`, e.g. half-years:

In [10]:
half = retail.groupby([retail.index.year, (retail.index.month - 1) // 6 + 1]).sum(
    min_count=6
)
print(half.tail(4))

date  date
2025  1       3626642.0
      2       3690807.0
2026  1       3787048.0
      2             NaN
Name: RSXFS, dtype: float64


## Monthly to quarterly, every option

The Empirical Macro Toolbox's `m2q` offers mean, median, start and end of
quarter. In pandas:

In [11]:
m2q = pd.DataFrame(
    {
        rule: unrate.resample("QS").agg(rule)
        for rule in ["mean", "median", "first", "last"]
    }
)
print(m2q.loc["2020"].round(1))

            mean  median  first  last
date                                 
2020-01-01   3.8     3.6    3.6   4.4
2020-04-01  13.0    13.2   14.8  11.0
2020-07-01   8.8     8.4   10.2   7.8
2020-10-01   6.8     6.7    6.9   6.7

## Pitfalls

* Aggregating growth rates is not aggregating levels. Aggregate the levels,
  then compute the growth rate.
* Averaging a ratio (e.g. debt/GDP) is not the ratio of averages. Aggregate
  numerator and denominator separately.
* Temporal aggregation changes the dynamics: averaging introduces an MA
  component and smooths turning points (Working, 1960). Keep this in mind
  before reading persistence off quarterly averages of monthly data.

## Summary

| The series is a… | Example | Aggregate with |
|---|---|---|
| **flow** (amount per period) | retail sales, claims, exports | `sum` |
| **flow at an annual rate** (SAAR) | US GDP, housing starts | `mean` |
| **stock** (level at a date) | debt, money stock, inventories | `last` |
| **average / rate / price / index** | unemployment rate, CPI, oil price | `mean` |

Count observations per period and drop periods that are incomplete.

## Exercises

1. Compute annual real GDP for the US (`GDPC1`) and for Portugal (`CLVMNACSCAB1GQPT`) for 2015–2025. One needs `mean` and the other `sum`: which is which, and how do you check this from the FRED units?
2. Aggregate payroll employment (`PAYEMS`) to quarters as the three-month average and as the end-of-quarter value for 2008–2010. Compute q/q growth of both: in which quarter does each series reach its trough, and why do they differ?
3. Convert weekly jobless claims (`ICSA`) to monthly totals with the naive week-ending rule and with the day-weighted function, for January–June 2020. How large is the misallocation in March and April 2020, and in which direction?
4. Build monthly Brent (`DCOILBRENTEU`) as the monthly average and as the end-of-month price for 2000–2019, and compute the first-order autocorrelation of the monthly log changes of each. Does the averaged series show the positive autocorrelation that Working (1960) predicts for an averaged random walk, and what does this imply for a VAR estimated on monthly averages?